# Import Packages

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import tqdm
import sklearn

In [2]:
dataset = sklearn.datasets.load_breast_cancer().data
standard_scaler = sklearn.preprocessing.StandardScaler()
dataset = standard_scaler.fit_transform(dataset)
dataset[:5]

array([[ 1.09706398e+00, -2.07333501e+00,  1.26993369e+00,
         9.84374905e-01,  1.56846633e+00,  3.28351467e+00,
         2.65287398e+00,  2.53247522e+00,  2.21751501e+00,
         2.25574689e+00,  2.48973393e+00, -5.65265059e-01,
         2.83303087e+00,  2.48757756e+00, -2.14001647e-01,
         1.31686157e+00,  7.24026158e-01,  6.60819941e-01,
         1.14875667e+00,  9.07083081e-01,  1.88668963e+00,
        -1.35929347e+00,  2.30360062e+00,  2.00123749e+00,
         1.30768627e+00,  2.61666502e+00,  2.10952635e+00,
         2.29607613e+00,  2.75062224e+00,  1.93701461e+00],
       [ 1.82982061e+00, -3.53632408e-01,  1.68595471e+00,
         1.90870825e+00, -8.26962447e-01, -4.87071673e-01,
        -2.38458552e-02,  5.48144156e-01,  1.39236330e-03,
        -8.68652457e-01,  4.99254601e-01, -8.76243603e-01,
         2.63326966e-01,  7.42401948e-01, -6.05350847e-01,
        -6.92926270e-01, -4.40780058e-01,  2.60162067e-01,
        -8.05450380e-01, -9.94437403e-02,  1.80592744e+

In [3]:
Xtrain, Xtest, ytrain, ytest = sklearn.model_selection.train_test_split(dataset, sklearn.datasets.load_breast_cancer().target, test_size=0.2)
Xtrain[:5], ytrain[:5]

(array([[-0.07875479,  0.07222088, -0.13547594, -0.1771566 , -0.67751502,
         -0.77778737, -0.94638513, -0.67036357, -0.93690009, -0.86014683,
         -0.22836214,  0.83667847, -0.2648707 , -0.23557405, -0.40167592,
         -0.81197099, -0.86479207, -0.77165362, -0.04870973, -0.55561486,
         -0.13236504,  0.37987822, -0.18947405, -0.23113586, -0.90164265,
         -0.89164551, -1.07780436, -0.84821623, -0.62730367, -0.82213894],
        [ 1.41231974,  1.62902878,  1.52943195,  1.35695235,  1.7890792 ,
          1.41679395,  1.31702506,  2.52731642, -0.64847556,  1.33855706,
          0.41082896,  3.07195015,  1.45288552,  0.58883002,  0.16801384,
          1.95735569, -0.34992981,  1.07607669,  1.21292827,  2.49460396,
          0.84092285,  1.14687248,  1.01387382,  0.73378242,  0.29946077,
          0.17452465, -0.13907293,  1.05815376, -0.95409536,  0.4479916 ],
        [-0.36844927, -0.8283541 , -0.3739672 , -0.4208962 , -0.42416605,
         -0.41031818, -0.38229028, -

In [4]:
Xtorch = torch.tensor(Xtrain, dtype=torch.float32)
ytorch = torch.tensor(ytrain, dtype=torch.float32)
Xtorch.shape, ytorch.shape

(torch.Size([455, 30]), torch.Size([455]))

In [5]:
Xtorch_test = torch.tensor(Xtest, dtype=torch.float32)
ytorch_test = torch.tensor(ytest, dtype=torch.float32)
Xtorch_test.shape, ytorch_test.shape

(torch.Size([114, 30]), torch.Size([114]))

In [6]:
features = sklearn.datasets.load_breast_cancer().feature_names
features

array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error',
       'fractal dimension error', 'worst radius', 'worst texture',
       'worst perimeter', 'worst area', 'worst smoothness',
       'worst compactness', 'worst concavity', 'worst concave points',
       'worst symmetry', 'worst fractal dimension'], dtype='<U23')

In [7]:
class LogisticRegressionModel(torch.nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = torch.nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

    def backward(self, loss):
        loss.backward()

In [8]:
l = LogisticRegressionModel(input_dim=Xtorch.shape[1])
loss = torch.nn.BCELoss()
optimizer = torch.optim.SGD(l.parameters(), lr=0.01)

for epoch in tqdm.tqdm(range(200)):
    optimizer.zero_grad()
    outputs = l(Xtorch)
    loss_value = loss(outputs, ytorch.unsqueeze(1).float())
    loss_value.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/200], Loss: {loss_value.item():.4f}')

100%|██████████| 200/200 [00:00<00:00, 1034.65it/s]

Epoch [10/200], Loss: 0.4515
Epoch [20/200], Loss: 0.3958
Epoch [30/200], Loss: 0.3563
Epoch [40/200], Loss: 0.3266
Epoch [50/200], Loss: 0.3034
Epoch [60/200], Loss: 0.2847
Epoch [70/200], Loss: 0.2694
Epoch [80/200], Loss: 0.2564
Epoch [90/200], Loss: 0.2454
Epoch [100/200], Loss: 0.2359
Epoch [110/200], Loss: 0.2275
Epoch [120/200], Loss: 0.2201
Epoch [130/200], Loss: 0.2135
Epoch [140/200], Loss: 0.2076
Epoch [150/200], Loss: 0.2023
Epoch [160/200], Loss: 0.1974
Epoch [170/200], Loss: 0.1930
Epoch [180/200], Loss: 0.1889
Epoch [190/200], Loss: 0.1851
Epoch [200/200], Loss: 0.1816


In [9]:
with torch.no_grad():
    outputs_test = l(Xtorch_test)
    predicted = (outputs_test >= 0.5).float()
    accuracy = (predicted == ytorch_test.unsqueeze(1)).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')
    tp = ((predicted == 1) & (ytorch_test.unsqueeze(1) == 1)).float().sum()
    fp = ((predicted == 1) & (ytorch_test.unsqueeze(1) == 0)).float().sum()
    fn = ((predicted == 0) & (ytorch_test.unsqueeze(1) == 1)).float().sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f'Precision: {precision.item():.4f}, Recall: {recall.item():.4f}')
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f'F1 Score: {f1_score.item():.4f}')

Test Accuracy: 0.9825
Precision: 0.9853, Recall: 0.9853
F1 Score: 0.9853


In [10]:
import pytikz as tz

In [16]:
pic = tz.Picture()

net = tz.NeuralNetwork(
    [len(features),1],
    titles="Logistic Network",
    arrows=True
)

pic.add(net)

pic.render("logistic.png")

PosixPath('logistic.png')

In [17]:
# From Scratch: Logistic Regression Implementation for Breast Cancer Classification

In [ ]:
dataset

array([[ 1.09706398, -2.07333501,  1.26993369, ...,  2.29607613,
         2.75062224,  1.93701461],
       [ 1.82982061, -0.35363241,  1.68595471, ...,  1.0870843 ,
        -0.24388967,  0.28118999],
       [ 1.57988811,  0.45618695,  1.56650313, ...,  1.95500035,
         1.152255  ,  0.20139121],
       ...,
       [ 0.70228425,  2.0455738 ,  0.67267578, ...,  0.41406869,
        -1.10454895, -0.31840916],
       [ 1.83834103,  2.33645719,  1.98252415, ...,  2.28998549,
         1.91908301,  2.21963528],
       [-1.80840125,  1.22179204, -1.81438851, ..., -1.74506282,
        -0.04813821, -0.75120669]], shape=(569, 30))

## feed Forward
$$
x = X^TW  + b
$$

$$
sigmoid = \frac{1}{1 + e^{x}}
$$

## loss
$$-\sum \frac{y * log(y_{pred}) + (1 - y) * log(1- y_{pred})}{m}$$

## backward
$$
m = y.shape\\
dw = \frac{np.dot(x^T(y_{pred} - y))}{m}\\
db = \sum \frac{y_{pred} - y}{m}
$$

## update parameters

$$
w -= \alpha * dw\\
bias -= \alpha * db
$$

In [19]:
def feed_forward(X, weights, bias):
    return 1 / (1 + np.exp(-(np.dot(X, weights) + bias)))

def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)) / m

def compute_gradients(X, y_true, y_pred):
    m = y_true.shape[0]
    dw = np.dot(X.T, (y_pred - y_true)) / m
    db = np.sum(y_pred - y_true) / m
    return dw, db

def update_parameters(weights, bias, dw, db, learning_rate):
    weights -= learning_rate * dw
    bias -= learning_rate * db
    return weights, bias

def train(X, y, weights, bias, learning_rate, epochs):
    for epoch in range(epochs):
        y_pred = feed_forward(X, weights, bias)
        loss = compute_loss(y, y_pred)
        dw, db = compute_gradients(X, y, y_pred)
        weights, bias = update_parameters(weights, bias, dw, db, learning_rate)
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss}")
    return weights, bias


In [20]:
x = dataset
y = sklearn.datasets.load_breast_cancer().target
x, y

(array([[ 1.09706398, -2.07333501,  1.26993369, ...,  2.29607613,
          2.75062224,  1.93701461],
        [ 1.82982061, -0.35363241,  1.68595471, ...,  1.0870843 ,
         -0.24388967,  0.28118999],
        [ 1.57988811,  0.45618695,  1.56650313, ...,  1.95500035,
          1.152255  ,  0.20139121],
        ...,
        [ 0.70228425,  2.0455738 ,  0.67267578, ...,  0.41406869,
         -1.10454895, -0.31840916],
        [ 1.83834103,  2.33645719,  1.98252415, ...,  2.28998549,
          1.91908301,  2.21963528],
        [-1.80840125,  1.22179204, -1.81438851, ..., -1.74506282,
         -0.04813821, -0.75120669]], shape=(569, 30)),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0

In [21]:
w = np.zeros(x.shape[1])
b = 0

learning_rate = 0.01

epochs = 1000

train(x, y, w, b, learning_rate, epochs)

Epoch 0, Loss: 0.6931471805599453
Epoch 100, Loss: 0.24891452349984458
Epoch 200, Loss: 0.18653598596242674
Epoch 300, Loss: 0.15829205252779793
Epoch 400, Loss: 0.1415708871440243
Epoch 500, Loss: 0.1303028685133958
Epoch 600, Loss: 0.12208830552790556
Epoch 700, Loss: 0.11577403072353368
Epoch 800, Loss: 0.11073248693276204
Epoch 900, Loss: 0.10659066946404366


(array([-0.38373657, -0.33684527, -0.37992606, -0.37875548, -0.14236926,
        -0.13853963, -0.2919467 , -0.38592821, -0.10443644,  0.16399939,
        -0.33478089, -0.00297918, -0.28831563, -0.3076237 ,  0.00511372,
         0.09146565,  0.07628203, -0.06223051,  0.06517495,  0.17600833,
        -0.45132578, -0.40879944, -0.43441022, -0.42532612, -0.30187862,
        -0.20785074, -0.2877202 , -0.40746938, -0.2783485 , -0.10111724]),
 np.float64(0.3288020102667734))

In [22]:
def accuracy(x, y, w, b):
    predictions = feed_forward(x, w, b)
    predictions = np.round(predictions)
    return np.mean(predictions == y)

def confusion_matrix(x, y, w, b):
    predictions = feed_forward(x, w, b)
    predictions = np.round(predictions)
    tp = np.sum((predictions == 1) & (y == 1))
    tn = np.sum((predictions == 0) & (y == 0))
    fp = np.sum((predictions == 1) & (y == 0))
    fn = np.sum((predictions == 0) & (y == 1))
    return np.array([[tp, fp], [fn, tn]])

def precision(x, y, w, b):
    predictions = feed_forward(x, w, b)
    tp, fp, fn, tn = confusion_matrix(x, y, w, b).ravel()
    return tp / (tp + fp) if (tp + fp) > 0 else 0

def recall(x, y, w, b):
    tp, fp, fn, tn = confusion_matrix(x, y, w, b).ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0

def f1_score(x, y, w, b):
    p = precision(x, y, w, b)
    r = recall(x, y, w, b)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

In [23]:
# Evaluation metrics for logistic regression
def evaluate_model(x, y, w, b):
    acc = accuracy(x, y, w, b)
    prec = precision(x, y, w, b)
    rec = recall(x, y, w, b)
    f1 = f1_score(x, y, w, b)
    cm = confusion_matrix(x, y, w, b)
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "confusion_matrix": cm
    }

evaluate_model(x, y, w, b)

{'accuracy': np.float64(0.9736379613356766),
 'precision': np.float64(0.9830508474576272),
 'recall': np.float64(0.9747899159663865),
 'f1_score': np.float64(0.9789029535864979),
 'confusion_matrix': array([[348,   6],
        [  9, 206]])}